# Yelp Restaurant Success — Analysis Notebook

**Brian Butler** — Probabilistic Machine Learning, Final Project, Spring 2026

This notebook consolidates all five analysis stages into a single runnable document.
Each section corresponds to one of the standalone scripts and can be run sequentially.

**Run order:** Data Prep → Baselines → Hierarchical → VBNN → Evaluate

**Python 3.11.7** | numpy 1.26.3 | pandas 2.2.0 | scikit-learn 1.4.0 | PyMC 5.10.3 | PyTorch 2.2.1 | Pyro 1.9.0

# Stage 1 — Data Preparation

Load Yelp JSON files, filter to 5 cities and 10 cuisines, engineer structured features, compute review embeddings with MiniLM, and split into train/val/test.

In [ ]:
from __future__ import annotations

import json
import pathlib
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
RAW_DIR = pathlib.Path("data/yelp_raw")
OUT_DIR = pathlib.Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CITIES = ["Philadelphia", "Tampa", "Indianapolis", "Nashville", "Tucson"]
CUISINES = [
    "American (Traditional)", "American (New)", "Italian", "Mexican",
    "Chinese", "Japanese", "Thai", "Indian", "Mediterranean", "Pizza",
]
MIN_REVIEWS = 20
REVIEWS_PER_BUSINESS = 30
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------

def load_jsonl(path: pathlib.Path, usecols: list[str] | None = None) -> pd.DataFrame:
    """Stream a Yelp JSONL file into a DataFrame, optionally projecting columns."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if usecols is not None:
                row = {k: row.get(k) for k in usecols}
            records.append(row)
    return pd.DataFrame.from_records(records)


def load_businesses() -> pd.DataFrame:
    cols = ["business_id", "name", "city", "state", "latitude", "longitude",
            "stars", "review_count", "is_open", "attributes", "categories", "hours"]
    return load_jsonl(RAW_DIR / "yelp_academic_dataset_business.json", cols)


def load_reviews(business_ids: set[str]) -> pd.DataFrame:
    cols = ["review_id", "business_id", "stars", "date", "text"]
    chunks = []
    with open(RAW_DIR / "yelp_academic_dataset_review.json", "r", encoding="utf-8") as f:
        buf = []
        for i, line in enumerate(f):
            row = json.loads(line)
            if row["business_id"] in business_ids:
                buf.append({k: row[k] for k in cols})
            if len(buf) >= 200_000:
                chunks.append(pd.DataFrame.from_records(buf))
                buf = []
        if buf:
            chunks.append(pd.DataFrame.from_records(buf))
    return pd.concat(chunks, ignore_index=True)


# ---------------------------------------------------------------------------
# Filtering
# ---------------------------------------------------------------------------

def is_restaurant(categories: str | None) -> bool:
    return isinstance(categories, str) and "Restaurants" in categories


def primary_cuisine(categories: str | None) -> str | None:
    if not isinstance(categories, str):
        return None
    tags = [t.strip() for t in categories.split(",")]
    for cuisine in CUISINES:
        if cuisine in tags:
            return cuisine
    return None


def filter_businesses(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["city"].isin(CITIES)].copy()
    df = df[df["categories"].apply(is_restaurant)]
    df["cuisine"] = df["categories"].apply(primary_cuisine)
    df = df.dropna(subset=["cuisine"])
    df = df[df["review_count"] >= MIN_REVIEWS]
    return df.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Structured features
# ---------------------------------------------------------------------------

def _safe_attr(attrs: dict | None, key: str, default=None):
    if not isinstance(attrs, dict):
        return default
    val = attrs.get(key, default)
    if isinstance(val, str) and val.lower() in {"true", "false"}:
        return val.lower() == "true"
    return val


def _parse_price(attrs: dict | None) -> float:
    val = _safe_attr(attrs, "RestaurantsPriceRange2")
    try:
        return float(val)
    except (TypeError, ValueError):
        return np.nan


def _parse_parking(attrs: dict | None) -> float:
    raw = _safe_attr(attrs, "BusinessParking")
    if isinstance(raw, str):
        try:
            raw = json.loads(raw.replace("'", '"').replace("False", "false").replace("True", "true"))
        except json.JSONDecodeError:
            return 0.0
    if not isinstance(raw, dict):
        return 0.0
    return float(sum(bool(v) for v in raw.values()))


def _weekend_hours(hours: dict | None) -> float:
    if not isinstance(hours, dict):
        return 0.0
    total = 0.0
    for day in ("Saturday", "Sunday"):
        span = hours.get(day)
        if not span:
            continue
        try:
            start, end = span.split("-")
            sh, sm = map(int, start.split(":"))
            eh, em = map(int, end.split(":"))
            delta = (eh + em / 60) - (sh + sm / 60)
            total += delta if delta > 0 else delta + 24
        except ValueError:
            continue
    return total


def _late_night(hours: dict | None) -> int:
    if not isinstance(hours, dict):
        return 0
    for span in hours.values():
        if not isinstance(span, str) or "-" not in span:
            continue
        try:
            end = span.split("-")[1]
            eh = int(end.split(":")[0])
            if eh >= 22 or eh <= 3:
                return 1
        except ValueError:
            continue
    return 0


def engineer_structured(df: pd.DataFrame, review_min_dates: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["price_range"] = df["attributes"].apply(_parse_price)
    out["price_range"] = out["price_range"].fillna(out["price_range"].median())
    out["outdoor_seating"] = df["attributes"].apply(
        lambda a: float(bool(_safe_attr(a, "OutdoorSeating")))
    )
    out["reservations"] = df["attributes"].apply(
        lambda a: float(bool(_safe_attr(a, "RestaurantsReservations")))
    )
    out["parking_score"] = df["attributes"].apply(_parse_parking)
    noise_map = {"quiet": 1, "average": 2, "loud": 3, "very_loud": 4}
    out["noise_level"] = df["attributes"].apply(
        lambda a: noise_map.get(str(_safe_attr(a, "NoiseLevel", "average")).strip("u'\""), 2)
    ).astype(float)
    out["weekend_open_hours"] = df["hours"].apply(_weekend_hours)
    out["late_night_open"] = df["hours"].apply(_late_night).astype(float)
    out["review_count_log"] = np.log1p(df["review_count"].astype(float))

    snapshot = pd.Timestamp("2024-11-01")
    age_days = (snapshot - pd.to_datetime(review_min_dates, errors="coerce")).dt.days
    out["age_years"] = (age_days / 365.25).fillna(age_days.median() / 365.25)

    return out


# ---------------------------------------------------------------------------
# Review embeddings
# ---------------------------------------------------------------------------

def sample_review_text(reviews: pd.DataFrame, n_per_business: int) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)

    def _sample(group: pd.DataFrame) -> pd.DataFrame:
        if len(group) <= n_per_business:
            return group
        idx = rng.choice(len(group), size=n_per_business, replace=False)
        return group.iloc[idx]

    return reviews.groupby("business_id", group_keys=False).apply(_sample)


def embed_reviews(reviews: pd.DataFrame, business_order: list[str]) -> np.ndarray:
    model = SentenceTransformer(EMBED_MODEL)
    embeddings = model.encode(
        reviews["text"].tolist(),
        batch_size=128,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    reviews = reviews.copy()
    reviews["embedding"] = list(embeddings)

    pooled = (
        reviews.groupby("business_id")["embedding"]
        .apply(lambda seq: np.mean(np.stack(seq.values), axis=0))
    )
    return np.stack([pooled.loc[bid] for bid in business_order])


# ---------------------------------------------------------------------------
# Target construction
# ---------------------------------------------------------------------------

@dataclass
class Targets:
    stars: np.ndarray
    is_open: np.ndarray
    composite: np.ndarray


def build_targets(df: pd.DataFrame) -> Targets:
    stars = df["stars"].astype(float).to_numpy()
    is_open = df["is_open"].astype(int).to_numpy()
    z_stars = (stars - stars.mean()) / stars.std()
    log_rc = np.log1p(df["review_count"].astype(float).to_numpy())
    z_log_rc = (log_rc - log_rc.mean()) / log_rc.std()
    composite = 0.5 * z_stars + 0.3 * z_log_rc + 0.2 * is_open
    return Targets(stars=stars, is_open=is_open, composite=composite)


# ---------------------------------------------------------------------------
# Splitting
# ---------------------------------------------------------------------------

def stratified_split(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    strat = df["city"] + "|" + df["cuisine"] + "|" + df["is_open"].astype(str)
    idx = np.arange(len(df))
    train_idx, rest_idx = train_test_split(
        idx, test_size=0.30, stratify=strat, random_state=SEED
    )
    val_idx, test_idx = train_test_split(
        rest_idx, test_size=0.50, stratify=strat.iloc[rest_idx], random_state=SEED
    )
    return train_idx, val_idx, test_idx


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------

### Run Stage 1 — Data Preparation

In [ ]:
businesses = filter_businesses(load_businesses())
print(f"Retained {len(businesses):,} restaurants across {len(CITIES)} cities")

reviews = load_reviews(set(businesses["business_id"]))
print(f"Loaded {len(reviews):,} reviews for these businesses")

min_dates = reviews.groupby("business_id")["date"].min()
businesses["first_review_date"] = businesses["business_id"].map(min_dates)

structured = engineer_structured(businesses, businesses["first_review_date"])
targets = build_targets(businesses)

sampled = sample_review_text(reviews, REVIEWS_PER_BUSINESS)
embeddings = embed_reviews(sampled, businesses["business_id"].tolist())
print(f"Embedding matrix shape: {embeddings.shape}")

train_idx, val_idx, test_idx = stratified_split(businesses)

scaler = StandardScaler().fit(structured.iloc[train_idx])
X_struct = scaler.transform(structured)

np.savez_compressed(
    OUT_DIR / "features.npz",
    X_struct=X_struct,
    X_embed=embeddings,
    stars=targets.stars,
    is_open=targets.is_open,
    composite=targets.composite,
    city=businesses["city"].to_numpy(),
    cuisine=businesses["cuisine"].to_numpy(),
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    struct_cols=np.array(structured.columns.tolist()),
)
print(f"Wrote {OUT_DIR / 'features.npz'}")

# Stage 2 — Frequentist Baselines

Ridge, Lasso, and Elastic Net for star-rating regression; L1 and L2 logistic regression for survival classification. Hyperparameters chosen by 5-fold CV.

In [ ]:
from __future__ import annotations

import pathlib

import numpy as np
import pandas as pd
from sklearn.linear_model import (
    ElasticNetCV,
    LassoCV,
    LogisticRegressionCV,
    RidgeCV,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

SEED = 42
PROC = pathlib.Path("data/processed")
OUT = pathlib.Path("results/baselines")
OUT.mkdir(parents=True, exist_ok=True)


# Load features and split indices from the preprocessed data
def load_data() -> dict:
    blob = np.load(PROC / "features.npz", allow_pickle=True)
    X = np.concatenate([blob["X_struct"], blob["X_embed"]], axis=1)
    return {
        "X": X,
        "stars": blob["stars"],
        "is_open": blob["is_open"],
        "train": blob["train_idx"],
        "val": blob["val_idx"],
        "test": blob["test_idx"],
    }


# Root mean squared error helper
def rmse(y, yhat) -> float:
    return float(np.sqrt(mean_squared_error(y, yhat)))


# Expected calibration error across probability bins
def expected_calibration_error(y_true, p_hat, n_bins: int = 15) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(p_hat, bins) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = idx == b
        if not mask.any():
            continue
        acc = y_true[mask].mean()
        conf = p_hat[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)


# ---------------------------------------------------------------------------
# Stars regression
# ---------------------------------------------------------------------------

# Fit Ridge, Lasso, ElasticNet with cross-validated alpha selection
def fit_regressors(X_train, y_train, X_test, y_test) -> pd.DataFrame:
    alphas = np.logspace(-4, 2, 40)
    results = []

    ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
    results.append(("Ridge", ridge.alpha_, None, ridge.predict(X_test)))

    lasso = LassoCV(alphas=alphas, cv=5, random_state=SEED, max_iter=20_000).fit(X_train, y_train)
    results.append(("Lasso", lasso.alpha_, None, lasso.predict(X_test)))

    enet = ElasticNetCV(
        alphas=alphas,
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        cv=5,
        random_state=SEED,
        max_iter=20_000,
    ).fit(X_train, y_train)
    results.append(("ElasticNet", enet.alpha_, enet.l1_ratio_, enet.predict(X_test)))

    rows = []
    for name, alpha, l1r, yhat in results:
        rows.append({
            "model": name,
            "alpha": alpha,
            "l1_ratio": l1r,
            "test_rmse": rmse(y_test, yhat),
            "test_mae": mean_absolute_error(y_test, yhat),
        })
    return pd.DataFrame(rows)


# Measure RMSE variance across multiple random training-order seeds
def seed_variance(X_train, y_train, X_test, y_test, seeds) -> pd.DataFrame:
    alphas = np.logspace(-4, 2, 40)
    per_seed = {"Ridge": [], "Lasso": [], "ElasticNet": []}
    for s in seeds:
        rng = np.random.default_rng(s)
        perm = rng.permutation(len(X_train))
        Xp, yp = X_train[perm], y_train[perm]

        per_seed["Ridge"].append(rmse(y_test, RidgeCV(alphas=alphas, cv=5).fit(Xp, yp).predict(X_test)))
        per_seed["Lasso"].append(rmse(y_test, LassoCV(
            alphas=alphas, cv=5, random_state=s, max_iter=20_000
        ).fit(Xp, yp).predict(X_test)))
        per_seed["ElasticNet"].append(rmse(y_test, ElasticNetCV(
            alphas=alphas, l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            cv=5, random_state=s, max_iter=20_000,
        ).fit(Xp, yp).predict(X_test)))

    rows = []
    for k, v in per_seed.items():
        v = np.array(v)
        rows.append({"model": k, "mean_rmse": v.mean(), "sem": v.std(ddof=1) / np.sqrt(len(v))})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Survival classification
# ---------------------------------------------------------------------------

# Fit L1 and L2 logistic regression with cross-validated C selection
def fit_classifiers(X_train, y_train, X_test, y_test) -> tuple[pd.DataFrame, np.ndarray]:
    Cs = np.logspace(-3, 2, 30)
    rows = []
    best_preds = None
    best_logloss = np.inf

    for penalty, solver in [("l2", "lbfgs"), ("l1", "liblinear")]:
        model = LogisticRegressionCV(
            Cs=Cs,
            cv=5,
            penalty=penalty,
            solver=solver,
            scoring="neg_log_loss",
            max_iter=5_000,
            random_state=SEED,
        ).fit(X_train, y_train)
        p = model.predict_proba(X_test)[:, 1]
        ll = log_loss(y_test, p)
        rows.append({
            "model": f"Logistic-{penalty.upper()}",
            "C": model.C_[0],
            "test_acc": accuracy_score(y_test, (p >= 0.5).astype(int)),
            "test_logloss": ll,
            "test_auc": roc_auc_score(y_test, p),
            "ece": expected_calibration_error(y_test, p),
        })
        if ll < best_logloss:
            best_logloss = ll
            best_preds = (p >= 0.5).astype(int)

    return pd.DataFrame(rows), best_preds


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Run Stage 2 — Frequentist Baselines

In [ ]:
d = load_data()
Xtr, Xte = d["X"][d["train"]], d["X"][d["test"]]
ytr_s, yte_s = d["stars"][d["train"]], d["stars"][d["test"]]
ytr_o, yte_o = d["is_open"][d["train"]], d["is_open"][d["test"]]

reg = fit_regressors(Xtr, ytr_s, Xte, yte_s)
reg.to_csv(OUT / "star_regression.csv", index=False)
print(reg.to_string(index=False))

sv = seed_variance(Xtr, ytr_s, Xte, yte_s, seeds=[42, 43, 44, 45, 46])
sv.to_csv(OUT / "star_regression_seed_variance.csv", index=False)
print(sv.to_string(index=False))

clf, best_preds = fit_classifiers(Xtr, ytr_o, Xte, yte_o)
clf.to_csv(OUT / "survival_classification.csv", index=False)
print(clf.to_string(index=False))

cm = confusion_matrix(yte_o, best_preds)
pd.DataFrame(cm, index=["actual_closed", "actual_open"],
             columns=["pred_closed", "pred_open"]).to_csv(OUT / "survival_confusion.csv")
print(cm)

# Stage 3 — Bayesian Hierarchical Model

Bayesian hierarchical regression with city and cuisine random intercepts (non-centred parameterisation), city-level random slopes on price range and weekend hours. Sampled with NUTS via PyMC 5.

In [ ]:
from __future__ import annotations

import pathlib

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm

SEED = 42
PROC = pathlib.Path("data/processed")
OUT = pathlib.Path("results/hierarchical")
OUT.mkdir(parents=True, exist_ok=True)


# Load structured features and group labels from preprocessed data
def load_data() -> dict:
    blob = np.load(PROC / "features.npz", allow_pickle=True)
    cols = list(blob["struct_cols"])
    X = blob["X_struct"]

    return {
        "X": X,
        "cols": cols,
        "stars": blob["stars"],
        "is_open": blob["is_open"],
        "city": blob["city"],
        "cuisine": blob["cuisine"],
        "train": blob["train_idx"],
        "test": blob["test_idx"],
    }


# Convert string labels to integer indices for PyMC indexing
def make_indices(labels: np.ndarray) -> tuple[np.ndarray, list[str]]:
    uniq = sorted(set(labels.tolist()))
    lookup = {name: i for i, name in enumerate(uniq)}
    return np.array([lookup[x] for x in labels]), uniq


# Build the hierarchical regression model with random intercepts and slopes
def build_model(d: dict) -> tuple[pm.Model, dict]:
    tr = d["train"]
    X = d["X"][tr]
    y = d["stars"][tr]
    z_features = ["price_range", "weekend_open_hours"]
    z_idx = [d["cols"].index(c) for c in z_features]
    Z = X[:, z_idx]

    city_idx, cities = make_indices(d["city"][tr])
    cuis_idx, cuisines = make_indices(d["cuisine"][tr])

    coords = {"city": cities, "cuisine": cuisines,
              "feature": d["cols"], "slope_feature": z_features}

    with pm.Model(coords=coords) as model:
        X_data = pm.Data("X", X)
        Z_data = pm.Data("Z", Z)
        city_data = pm.Data("city_idx", city_idx)
        cuis_data = pm.Data("cuisine_idx", cuis_idx)

        # Global intercept and fixed-effect coefficients
        alpha0 = pm.Normal("alpha0", 0.0, 5.0)
        beta = pm.Normal("beta", 0.0, 2.5, dims="feature")

        # Non-centred city random intercepts
        tau_city = pm.HalfNormal("tau_city", 1.0)
        alpha_city_raw = pm.Normal("alpha_city_raw", 0.0, 1.0, dims="city")
        alpha_city = pm.Deterministic("alpha_city", tau_city * alpha_city_raw, dims="city")

        # Non-centred cuisine random intercepts
        tau_cuis = pm.HalfNormal("tau_cuisine", 1.0)
        alpha_cuis_raw = pm.Normal("alpha_cuis_raw", 0.0, 1.0, dims="cuisine")
        alpha_cuis = pm.Deterministic("alpha_cuisine",
                                      tau_cuis * alpha_cuis_raw, dims="cuisine")

        # Non-centred city-level random slopes on price_range and weekend hours
        tau_slope = pm.HalfNormal("tau_slope", 1.0)
        gamma_raw = pm.Normal("gamma_raw", 0.0, 1.0, dims=("city", "slope_feature"))
        gamma_city = pm.Deterministic("gamma_city",
                                      tau_slope * gamma_raw, dims=("city", "slope_feature"))

        # Linear predictor combining all components
        mu = (
            alpha0
            + alpha_city[city_data]
            + alpha_cuis[cuis_data]
            + pm.math.dot(X_data, beta)
            + pm.math.sum(Z_data * gamma_city[city_data], axis=1)
        )

        sigma = pm.HalfNormal("sigma", 1.0)
        pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    meta = {
        "cities": cities,
        "cuisines": cuisines,
        "z_features": z_features,
        "z_idx": z_idx,
        "train_city_idx": city_idx,
        "train_cuis_idx": cuis_idx,
    }
    return model, meta


# Run NUTS sampling with 4 chains and return the InferenceData
def sample(model: pm.Model) -> az.InferenceData:
    with model:
        idata = pm.sample(
            draws=2000,
            tune=2000,
            chains=4,
            target_accept=0.95,
            max_treedepth=12,
            random_seed=SEED,
            init="jitter+adapt_diag",
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": False},
        )
        idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEED))
    return idata


# Compute convergence diagnostics (Rhat, ESS, divergences)
def diagnostics(idata: az.InferenceData) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary = az.summary(
        idata,
        var_names=["alpha0", "beta", "alpha_city", "alpha_cuisine",
                   "gamma_city", "tau_city", "tau_cuisine", "tau_slope", "sigma"],
        round_to=4,
    )
    diverging = int(idata.sample_stats["diverging"].sum())
    summary_tail = pd.DataFrame({
        "worst_rhat": [summary["r_hat"].max()],
        "min_ess_bulk": [summary["ess_bulk"].min()],
        "min_ess_tail": [summary["ess_tail"].min()],
        "divergences": [diverging],
    })
    return summary, summary_tail


# Posterior predictive mean for the held-out test set
def predict_test(idata, d, meta) -> np.ndarray:
    post = idata.posterior

    X_test = d["X"][d["test"]]
    Z_test = X_test[:, meta["z_idx"]]

    # Map test labels into training index space; unseen groups default to 0
    city_lookup = {c: i for i, c in enumerate(meta["cities"])}
    cuis_lookup = {c: i for i, c in enumerate(meta["cuisines"])}
    city_test = np.array([city_lookup.get(c, -1) for c in d["city"][d["test"]]])
    cuis_test = np.array([cuis_lookup.get(c, -1) for c in d["cuisine"][d["test"]]])

    # Average across chains and draws to get point estimates
    alpha0 = post["alpha0"].mean(("chain", "draw")).values
    beta = post["beta"].mean(("chain", "draw")).values
    ac = post["alpha_city"].mean(("chain", "draw")).values
    aq = post["alpha_cuisine"].mean(("chain", "draw")).values
    gc = post["gamma_city"].mean(("chain", "draw")).values

    mu = alpha0 + X_test @ beta
    mu += np.where(city_test >= 0, ac[city_test.clip(min=0)], 0.0)
    mu += np.where(cuis_test >= 0, aq[cuis_test.clip(min=0)], 0.0)
    mu += np.where(
        city_test >= 0,
        np.sum(Z_test * gc[city_test.clip(min=0)], axis=1),
        0.0,
    )
    return mu

### Run Stage 3 — Bayesian Hierarchical Model

In [ ]:
d = load_data()
model, meta = build_model(d)
idata = sample(model)
idata.to_netcdf(OUT / "trace.nc")

summary, tail = diagnostics(idata)
summary.to_csv(OUT / "posterior_summary.csv")
tail.to_csv(OUT / "diagnostics.csv", index=False)
print(tail.to_string(index=False))

yhat = predict_test(idata, d, meta)
yte = d["stars"][d["test"]]
rmse = float(np.sqrt(np.mean((yte - yhat) ** 2)))
pd.DataFrame({"metric": ["test_rmse"], "value": [rmse]}).to_csv(
    OUT / "test_performance.csv", index=False
)
print(f"Hierarchical test RMSE: {rmse:.4f}")

# Stage 4 — Variational Bayesian Neural Network

Two hidden layers of width 128 with mean-field Gaussian variational posterior. Heteroscedastic Gaussian head for star rating and Bernoulli head for survival. Trained via SVI with KL annealing.

In [ ]:
from __future__ import annotations

import pathlib

import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
import torch
import torch.nn as nn
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoNormal
from pyro.nn import PyroModule, PyroSample

SEED = 42
PROC = pathlib.Path("data/processed")
OUT = pathlib.Path("results/vbnn")
OUT.mkdir(parents=True, exist_ok=True)

HIDDEN = 128
EPOCHS = 150
BATCH = 256
LR = 1e-3
KL_ANNEAL_EPOCHS = 20
POSTERIOR_SAMPLES = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
pyro.set_rng_seed(SEED)


# Load features and targets, cast to float32 for PyTorch
def load_data() -> dict:
    blob = np.load(PROC / "features.npz", allow_pickle=True)
    X = np.concatenate([blob["X_struct"], blob["X_embed"]], axis=1).astype(np.float32)
    return {
        "X": X,
        "stars": blob["stars"].astype(np.float32),
        "is_open": blob["is_open"].astype(np.float32),
        "train": blob["train_idx"],
        "val": blob["val_idx"],
        "test": blob["test_idx"],
    }


class BayesianLinear(PyroModule):
    """Dense layer with a standard-normal prior on every weight."""

    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.weight = PyroSample(
            dist.Normal(torch.zeros(out_dim, in_dim), torch.ones(out_dim, in_dim)).to_event(2)
        )
        self.bias = PyroSample(
            dist.Normal(torch.zeros(out_dim), torch.ones(out_dim)).to_event(1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight.T + self.bias


class VBNN(PyroModule):
    """Two-hidden-layer BNN with heteroscedastic Gaussian + Bernoulli heads."""

    def __init__(self, in_dim: int, hidden: int = HIDDEN) -> None:
        super().__init__()
        self.fc1 = BayesianLinear(in_dim, hidden)
        self.fc2 = BayesianLinear(hidden, hidden)
        self.mean_head = BayesianLinear(hidden, 1)
        self.logvar_head = BayesianLinear(hidden, 1)
        self.logit_head = BayesianLinear(hidden, 1)
        self.act = nn.ReLU()

    def trunk(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.fc2(self.act(self.fc1(x))))

    def forward(self, x: torch.Tensor,
                y_stars: torch.Tensor | None = None,
                y_open: torch.Tensor | None = None,
                kl_weight: float = 1.0) -> torch.Tensor:
        h = self.trunk(x)
        mu = self.mean_head(h).squeeze(-1)
        log_var = self.logvar_head(h).squeeze(-1)
        sigma = torch.exp(0.5 * log_var).clamp(min=1e-3, max=3.0)
        logit = self.logit_head(h).squeeze(-1)

        with pyro.plate("data", x.shape[0]):
            if y_stars is not None:
                pyro.sample("stars", dist.Normal(mu, sigma), obs=y_stars)
            if y_open is not None:
                pyro.sample("is_open", dist.Bernoulli(logits=logit), obs=y_open)
        return mu, sigma, logit


# Wrap numpy arrays into a DataLoader for mini-batch training
def make_loader(X, ys, yo, batch: int = BATCH, shuffle: bool = True):
    X_t = torch.from_numpy(X)
    ys_t = torch.from_numpy(ys)
    yo_t = torch.from_numpy(yo)
    ds = torch.utils.data.TensorDataset(X_t, ys_t, yo_t)
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=shuffle,
                                       num_workers=0, drop_last=False)


# Train the model via SVI with KL annealing over the first 20 epochs
def train(model: VBNN, guide, Xtr, ys_tr, yo_tr) -> list[float]:
    optim = pyro.optim.Adam({"lr": LR})
    svi = SVI(model, guide, optim, loss=Trace_ELBO())
    loader = make_loader(Xtr, ys_tr, yo_tr)
    history = []

    for epoch in range(EPOCHS):
        kl_weight = min(1.0, (epoch + 1) / KL_ANNEAL_EPOCHS)
        running = 0.0
        n = 0
        for xb, ysb, yob in loader:
            loss = svi.step(xb, ysb, yob, kl_weight=kl_weight)
            running += loss
            n += xb.shape[0]
        history.append(running / n)
        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1:3d}  neg_elbo/N = {history[-1]:.4f}  kl_w = {kl_weight:.2f}")
    return history


# ---------------------------------------------------------------------------
# Inference helpers
# ---------------------------------------------------------------------------

# Draw posterior samples and decompose predictive uncertainty
def posterior_predict(model: VBNN, guide, X: np.ndarray, n_samples: int = POSTERIOR_SAMPLES):
    X_t = torch.from_numpy(X)
    mus, sigmas, logits = [], [], []
    for _ in range(n_samples):
        guide_trace = pyro.poutine.trace(guide).get_trace(X_t)
        replayed = pyro.poutine.replay(model, trace=guide_trace)
        mu, sigma, logit = replayed(X_t)
        mus.append(mu.detach().numpy())
        sigmas.append(sigma.detach().numpy())
        logits.append(logit.detach().numpy())

    mus = np.stack(mus)
    sigmas = np.stack(sigmas)
    probs = 1.0 / (1.0 + np.exp(-np.stack(logits)))

    # Decompose into aleatoric (average data noise) and epistemic (weight uncertainty)
    mean_pred = mus.mean(axis=0)
    aleatoric = (sigmas ** 2).mean(axis=0)
    epistemic = mus.var(axis=0)
    total_std = np.sqrt(aleatoric + epistemic)
    return {
        "mean": mean_pred,
        "aleatoric_var": aleatoric,
        "epistemic_var": epistemic,
        "total_std": total_std,
        "prob_open": probs.mean(axis=0),
        "prob_open_std": probs.std(axis=0),
    }


# Expected calibration error across probability bins
def expected_calibration_error(y_true, p_hat, n_bins: int = 15) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(p_hat, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += (m.sum() / n) * abs(y_true[m].mean() - p_hat[m].mean())
    return float(ece)

### Run Stage 4 — Variational Bayesian Neural Network

In [ ]:
d = load_data()
Xtr = d["X"][d["train"]]
Xte = d["X"][d["test"]]
ys_tr = d["stars"][d["train"]]
yo_tr = d["is_open"][d["train"]]
ys_te = d["stars"][d["test"]]
yo_te = d["is_open"][d["test"]]

pyro.clear_param_store()
model = VBNN(in_dim=Xtr.shape[1])
guide = AutoNormal(model, init_scale=0.05)

history = train(model, guide, Xtr, ys_tr, yo_tr)
pd.DataFrame({"epoch": np.arange(1, len(history) + 1),
              "neg_elbo_per_obs": history}).to_csv(OUT / "elbo_history.csv", index=False)

preds = posterior_predict(model, guide, Xte)
rmse = float(np.sqrt(np.mean((ys_te - preds["mean"]) ** 2)))
pred_open = (preds["prob_open"] >= 0.5).astype(int)
acc = float((pred_open == yo_te).mean())

# Manual binary cross-entropy with clipping to avoid log(0)
p_clipped = preds["prob_open"].clip(1e-7, 1 - 1e-7)
ll = float(-np.mean(
    yo_te * np.log(p_clipped)
    + (1 - yo_te) * np.log(1 - p_clipped)
))
ece = expected_calibration_error(yo_te, preds["prob_open"])

summary = pd.DataFrame([{
    "test_rmse": rmse,
    "test_acc": acc,
    "test_logloss": ll,
    "ece": ece,
    "mean_aleatoric_std": float(np.sqrt(preds["aleatoric_var"]).mean()),
    "mean_epistemic_std": float(np.sqrt(preds["epistemic_var"]).mean()),
}])
summary.to_csv(OUT / "test_performance.csv", index=False)
print(summary.to_string(index=False))

np.savez_compressed(
    OUT / "predictions.npz",
    mean=preds["mean"],
    aleatoric_var=preds["aleatoric_var"],
    epistemic_var=preds["epistemic_var"],
    prob_open=preds["prob_open"],
    prob_open_std=preds["prob_open_std"],
)

# Stage 5 — Consolidated Evaluation

Head-to-head comparison table, uncertainty-vs-error analysis, variance decomposition from the hierarchical posterior, embedding ablation, and top fixed effects summary.

In [ ]:
from __future__ import annotations

import pathlib

import arviz as az
import numpy as np
import pandas as pd

PROC = pathlib.Path("data/processed")
BASE = pathlib.Path("results/baselines")
HIER = pathlib.Path("results/hierarchical")
VBNN = pathlib.Path("results/vbnn")
OUT = pathlib.Path("results/final")
OUT.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------

# Load test-set targets and metadata for evaluation
def load_targets() -> dict:
    blob = np.load(PROC / "features.npz", allow_pickle=True)
    cols = list(blob["struct_cols"])
    return {
        "stars": blob["stars"],
        "is_open": blob["is_open"],
        "test": blob["test_idx"],
        "city": blob["city"],
        "cuisine": blob["cuisine"],
        "struct_cols": cols,
        "X_struct": blob["X_struct"],
        "review_count_log": blob["X_struct"][:, cols.index("review_count_log")],
    }


# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------

# Build the headline model-vs-model comparison from saved CSV results
def headline_table() -> pd.DataFrame:
    base_reg = pd.read_csv(BASE / "star_regression.csv")
    base_clf = pd.read_csv(BASE / "survival_classification.csv")
    hier = pd.read_csv(HIER / "test_performance.csv")
    vbnn = pd.read_csv(VBNN / "test_performance.csv")

    rows = [
        {"model": "Ridge",
         "test_rmse": base_reg.loc[base_reg.model == "Ridge", "test_rmse"].iloc[0],
         "test_logloss": base_clf.loc[base_clf.model == "Logistic-L2", "test_logloss"].iloc[0],
         "ece": base_clf.loc[base_clf.model == "Logistic-L2", "ece"].iloc[0]},
        {"model": "ElasticNet",
         "test_rmse": base_reg.loc[base_reg.model == "ElasticNet", "test_rmse"].iloc[0],
         "test_logloss": base_clf.loc[base_clf.model == "Logistic-L1", "test_logloss"].iloc[0],
         "ece": base_clf.loc[base_clf.model == "Logistic-L1", "ece"].iloc[0]},
        {"model": "Hierarchical",
         "test_rmse": hier["value"].iloc[0],
         "test_logloss": np.nan,
         "ece": np.nan},
        {"model": "VBNN",
         "test_rmse": vbnn["test_rmse"].iloc[0],
         "test_logloss": vbnn["test_logloss"].iloc[0],
         "ece": vbnn["ece"].iloc[0]},
    ]
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Uncertainty-vs-error analysis
# ---------------------------------------------------------------------------

# Correlate epistemic uncertainty with prediction error and build risk bands
def uncertainty_vs_error(t: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    preds = np.load(VBNN / "predictions.npz")
    yte_stars = t["stars"][t["test"]]
    yte_open = t["is_open"][t["test"]]

    abs_err_stars = np.abs(yte_stars - preds["mean"])
    epi_std = np.sqrt(preds["epistemic_var"])
    corr_stars = float(np.corrcoef(epi_std, abs_err_stars)[0, 1])

    # Bin predicted probability of being open and check calibration per band
    p_open = preds["prob_open"]
    bands = np.array([0.0, 0.3, 0.5, 0.7, 0.8, 1.01])
    band_idx = np.digitize(p_open, bands) - 1
    band_idx = np.clip(band_idx, 0, len(bands) - 2)
    rows = []
    for b in range(len(bands) - 1):
        m = band_idx == b
        if not m.any():
            continue
        rows.append({
            "band": f"[{bands[b]:.1f}, {bands[b+1]:.2f})",
            "n": int(m.sum()),
            "mean_prob_open": float(p_open[m].mean()),
            "observed_open_rate": float(yte_open[m].mean()),
            "mean_epistemic_sd": float(preds["prob_open_std"][m].mean()),
        })
    return pd.DataFrame([{"metric": "corr_epistemic_abs_err_stars", "value": corr_stars}]), pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Variance decomposition from the hierarchical posterior
# ---------------------------------------------------------------------------

# Decompose total variance into city, cuisine, and residual components
def variance_decomposition() -> pd.DataFrame:
    idata = az.from_netcdf(HIER / "trace.nc")
    tau_city = float(idata.posterior["tau_city"].mean())
    tau_cuis = float(idata.posterior["tau_cuisine"].mean())
    sigma = float(idata.posterior["sigma"].mean())
    total = tau_city ** 2 + tau_cuis ** 2 + sigma ** 2
    return pd.DataFrame([{
        "tau_city": tau_city,
        "tau_cuisine": tau_cuis,
        "sigma": sigma,
        "icc_city": tau_city ** 2 / total,
        "icc_cuisine": tau_cuis ** 2 / total,
        "share_residual": sigma ** 2 / total,
    }])


# ---------------------------------------------------------------------------
# Embedding ablation: fit Ridge with and without the 384-dim block
# ---------------------------------------------------------------------------

# Compare Ridge RMSE with structured features only vs. with embeddings added
def embedding_ablation() -> pd.DataFrame:
    from sklearn.linear_model import RidgeCV

    blob = np.load(PROC / "features.npz", allow_pickle=True)
    Xs = blob["X_struct"]
    Xe = blob["X_embed"]
    stars = blob["stars"]
    tr, te = blob["train_idx"], blob["test_idx"]
    alphas = np.logspace(-3, 2, 25)

    rows = []
    for name, X in [("structured_only", Xs),
                    ("structured_plus_embeddings", np.concatenate([Xs, Xe], axis=1))]:
        model = RidgeCV(alphas=alphas, cv=5).fit(X[tr], stars[tr])
        yhat = model.predict(X[te])
        rows.append({
            "feature_set": name,
            "test_rmse": float(np.sqrt(np.mean((stars[te] - yhat) ** 2))),
            "alpha": float(model.alpha_),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Business-question summary
# ---------------------------------------------------------------------------

# Extract the strongest fixed effects from the hierarchical posterior
def top_fixed_effects(k: int = 6) -> pd.DataFrame:
    idata = az.from_netcdf(HIER / "trace.nc")
    summary = az.summary(idata, var_names=["beta"], round_to=4)
    summary["abs_mean"] = summary["mean"].abs()
    summary = summary.sort_values("abs_mean", ascending=False).head(k)
    return summary[["mean", "sd", "hdi_3%", "hdi_97%", "ess_bulk", "r_hat"]]


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Run Stage 5 — Consolidated Evaluation

In [ ]:
t = load_targets()

head = headline_table()
head.to_csv(OUT / "headline_comparison.csv", index=False)
print("\n=== Headline comparison ===")
print(head.to_string(index=False))

unc_scalar, risk_bands = uncertainty_vs_error(t)
unc_scalar.to_csv(OUT / "uncertainty_correlation.csv", index=False)
risk_bands.to_csv(OUT / "survival_risk_bands.csv", index=False)
print("\n=== Risk bands ===")
print(risk_bands.to_string(index=False))

var = variance_decomposition()
var.to_csv(OUT / "variance_decomposition.csv", index=False)
print("\n=== Variance decomposition ===")
print(var.to_string(index=False))

abl = embedding_ablation()
abl.to_csv(OUT / "embedding_ablation.csv", index=False)
print("\n=== Embedding ablation ===")
print(abl.to_string(index=False))

top = top_fixed_effects()
top.to_csv(OUT / "top_fixed_effects.csv")
print("\n=== Top fixed effects ===")
print(top.to_string())